# 01 — Data Exploration
Explore all data sources: parking occupancy, weather, events, permits, holidays, road closures.

In [ ]:
import pandas as pd
import json
from pathlib import Path

DATA = Path('../data')

## 1. Annual Parking Study (2014–2019)
Manual occupancy counts — used as validation baseline.

In [ ]:
parking = pd.read_csv(DATA / 'Annual_Parking_Study_Data_20250220.csv', low_memory=False)
print(f'Shape: {parking.shape}')
print(f'Date range: {parking["Date Time"].min()} → {parking["Date Time"].max()}')
print(f'Study Areas: {parking["Study_Area"].nunique()} unique')
parking.head()

In [ ]:
print('Records per Study Year:')
print(parking['Study Year'].value_counts().sort_index())
print()
print('Null counts:')
print(parking.isnull().sum())

## 2. Live Paid Parking Occupancy (Last 30 Days)
Real meter transaction data — primary live data source.

In [ ]:
live = pd.read_csv(DATA / 'live_parking_occupancy.csv', parse_dates=['occupancy_date'])
print(f'Shape: {live.shape}')
print(f'Date range: {live["occupancy_date"].min().date()} → {live["occupancy_date"].max().date()}')
print(f'Unique blockfaces: {live["blockfacename"].nunique()}')
live.describe()

In [ ]:
import matplotlib.pyplot as plt

daily_occ = live.groupby('occupancy_date')['occupancy_rate'].mean()
plt.figure(figsize=(12, 4))
plt.plot(daily_occ.index, daily_occ.values, color='steelblue')
plt.title('Average Daily Occupancy Rate — Last 30 Days')
plt.ylabel('Occupancy Rate')
plt.xlabel('Date')
plt.tight_layout()
plt.savefig('../visualizations/parking/daily_occupancy_live.png', dpi=150)
plt.show()

## 3. Weather Data (2014–present)
5 Seattle neighborhoods, hourly: temperature, precipitation, wind speed.

In [ ]:
weather = pd.read_csv(DATA / 'processed_weather_data.csv', parse_dates=['timestamp'])
print(f'Shape: {weather.shape}')
print(f'Date range: {weather["timestamp"].min()} → {weather["timestamp"].max()}')
print(f'Regions: {weather["region"].unique()}')
weather.describe()

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
weather['temperature'].hist(bins=40, ax=axes[0], color='tomato'); axes[0].set_title('Temperature (°C)')
weather['precipitation'].hist(bins=40, ax=axes[1], color='steelblue'); axes[1].set_title('Precipitation (mm)')
weather['wind_speed'].hist(bins=40, ax=axes[2], color='seagreen'); axes[2].set_title('Wind Speed (m/s)')
plt.tight_layout()
plt.savefig('../visualizations/weather/weather_distributions.png', dpi=150)
plt.show()

## 4. Events, Permits, Holidays, Closures

In [ ]:
events   = pd.read_csv(DATA / 'seattle_events.csv', parse_dates=['event_date'])
permits  = pd.read_csv(DATA / 'seattle_event_permits.csv', parse_dates=['event_start_date'])
holidays = pd.read_csv(DATA / 'seattle_holidays.csv', parse_dates=['date'])
closures = pd.read_csv(DATA / 'seattle_road_closures.csv', parse_dates=['start_date'])

print(f'Events:   {len(events):,} rows  | Venues: {events["venue"].value_counts().to_dict()}')
print(f'Permits:  {len(permits):,} rows  | Types: {permits["permit_type"].value_counts().head(3).to_dict()}')
print(f'Holidays: {len(holidays):,} rows  | Types: {holidays["holiday_type"].value_counts().to_dict()}')
print(f'Closures: {len(closures):,} rows  | Types: {closures["permit_type"].value_counts().head(3).to_dict()}')

In [ ]:
print('Upcoming events (next 60 days):')
events.sort_values('event_date')[['event_name', 'venue', 'event_date', 'genre']].head(20)

## 5. Aggregated Feature Set

In [ ]:
features = pd.read_csv(DATA / 'features.csv', parse_dates=['hour'])
print(f'Shape: {features.shape}')
print(f'Columns: {features.columns.tolist()}')
features.describe()

## Data Coverage Summary

| Source | Records | Date Range | Status |
|--------|---------|------------|--------|
| Annual Parking Study | 167,799 | 2014–2019 | ✅ Validation baseline |
| Live Paid Occupancy | 153,324 | Last 30 days | ✅ Daily updated |
| Weather (5 regions) | 238,920 | 2014–present | ✅ Daily updated |
| Events (3 venues) | 119 | Next 60 days | ✅ Daily updated |
| City Event Permits | 2,018 | 2014–present | ✅ Incremental |
| Holidays | 298 | 2012–2027 | ✅ Static library |
| Road Closures | 140 | Active | ✅ Daily updated |
| Feature Set | 364 | Last 30 days | ⚠️ Needs historical backfill |